# ComfyUI バックアップ（Flux.2 Klein用）


In [ ]:
import os, subprocess, zipfile, requests as _req
from pathlib import Path
from datetime import datetime, timezone, timedelta

WORK_DIR     = "/workspace/runpod-slim"
COMFY_DIR    = f"{WORK_DIR}/ComfyUI"
WORKFLOWS_DIR = f"{COMFY_DIR}/user/default/workflows"
OUTPUT_DIR   = f"{COMFY_DIR}/output"

# ===== Slack Webhook URLを.envから読み込み =====
ENV_FILE = f"{WORK_DIR}/.env"
def load_env():
    env = {}
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env()
SLACK_WEBHOOK_URL = env.get("SLACK_WEBHOOK_URL", "")

# ===== ファイル名のラベル設定 =====
# 任意のラベル（空欄でもOK）を指定すると、ファイル名末尾に追加されます。
# 例: LABEL = "ollama-migration"  → comfyui_backup_flux2_2026-05-27_18-54_ollama-migration.zip
#     LABEL = ""                  → comfyui_backup_flux2_2026-05-27_18-54.zip
LABEL = ""

# ===== ファイル名生成 =====
JST = timezone(timedelta(hours=9))
timestamp = datetime.now(JST).strftime("%Y-%m-%d_%H-%M")
if LABEL.strip():
    # ラベルをファイル名として安全な文字に変換（英数字とハイフン・アンダースコア以外は-に置換）
    safe_label = "".join(c if c.isalnum() or c in "-_" else "-" for c in LABEL.strip())
    # 連続するハイフンを1つにまとめる + 先頭末尾のハイフン削除
    while "--" in safe_label:
        safe_label = safe_label.replace("--", "-")
    safe_label = safe_label.strip("-")
    suffix = f"{timestamp}_{safe_label}" if safe_label else timestamp
else:
    suffix = timestamp

MAIN_ZIP   = f"{WORK_DIR}/comfyui_backup_flux2_{suffix}.zip"
OUTPUT_ZIP = f"{WORK_DIR}/comfyui_output_flux2_{suffix}.zip"
print(f"📝 ファイル名: comfyui_backup_flux2_{suffix}.zip")

# ===== カスタムノードURLを記録 =====
custom_nodes_dir = Path(f"{COMFY_DIR}/custom_nodes")
urls = []
if custom_nodes_dir.exists():
    for d in sorted(custom_nodes_dir.iterdir()):
        if d.is_dir() and (d / ".git").exists():
            try:
                url = subprocess.run(
                    ["git", "-C", str(d), "config", "--get", "remote.origin.url"],
                    capture_output=True, text=True, check=True
                ).stdout.strip()
                if url:
                    urls.append(url)
            except subprocess.CalledProcessError:
                pass

custom_nodes_txt = f"{WORK_DIR}/custom_nodes.txt"
with open(custom_nodes_txt, "w") as f:
    f.write("\n".join(sorted(set(urls))) + "\n")
print(f"📝 custom_nodes.txt: {len(urls)} 個のノード記録")

# ===== メインzip（output以外）=====
print(f"\n📦 メインzip作成中: {os.path.basename(MAIN_ZIP)}")

# zipに含めるアイテム（パス, アーカイブ内パス）
WILDCARD_DIR = f"{COMFY_DIR}/custom_nodes/ComfyUI_AB_Wildcard/wildcards"

items = [
    # 設定ファイル類
    (f"{WORK_DIR}/setup_flux2_klein.ipynb",             "setup_flux2_klein.ipynb"),
    (f"{WORK_DIR}/backup.ipynb",                        "backup.ipynb"),
    (f"{WORK_DIR}/.env",                                ".env"),
    (custom_nodes_txt,                                  "custom_nodes.txt"),
    # ダウンロード関連
    (f"{WORK_DIR}/download_list.txt",                   "download_list.txt"),
    (f"{WORK_DIR}/download_ui.ipynb",                   "download_ui.ipynb"),
    (f"{WORK_DIR}/download_extra.ipynb",                "download_extra.ipynb"),
    # HTML
    (f"{WORK_DIR}/comfyui_mobile.html",                 "comfyui_mobile.html"),
    # world_setting
    (f"{WORK_DIR}/world_setting.txt",                   "world_setting.txt"),
    # ワークフロー
    (f"{WORK_DIR}/flux2_klein_16GB_workflow_v2ollama.json", "flux2_klein_16GB_workflow_v2ollama.json"),
    (f"{WORK_DIR}/flux2_klein_24GB_workflow_v2ollama.json", "flux2_klein_24GB_workflow_v2ollama.json"),
    (f"{WORK_DIR}/flux2_klein_32GB_workflow_v2ollama.json", "flux2_klein_32GB_workflow_v2ollama.json"),
    (f"{WORK_DIR}/flux2_klein_48GB_workflow_v2ollama.json", "flux2_klein_48GB_workflow_v2ollama.json"),
    # ComfyUI内の重要ファイル
    (WORKFLOWS_DIR,                                     "ComfyUI/user/default/workflows"),
    (f"{COMFY_DIR}/user/default/comfy.settings.json",   "ComfyUI/user/default/comfy.settings.json"),
    # wildcards
    (WILDCARD_DIR,                                      "wildcards"),
]

with zipfile.ZipFile(MAIN_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for src, arc in items:
        src_path = Path(src)
        if not src_path.exists():
            print(f"  ⚠️  スキップ: {arc}")
            continue
        if src_path.is_file():
            zf.write(src_path, arc)
            print(f"  ✅ {arc}")
        else:
            count = 0
            for p in src_path.rglob("*"):
                if p.is_file():
                    rel = p.relative_to(src_path)
                    zf.write(p, f"{arc}/{rel}")
                    count += 1
            print(f"  ✅ {arc}/ ({count} ファイル)")

main_size_mb = os.path.getsize(MAIN_ZIP) / 1e6
print(f"\n   📦 {MAIN_ZIP}")
print(f"   サイズ: {main_size_mb:.1f} MB")

# ===== outputzip（生成画像のみ）=====
output_size_mb = 0
if os.path.exists(OUTPUT_DIR) and any(Path(OUTPUT_DIR).rglob("*")):
    print(f"\n🖼️  outputzip作成中: {os.path.basename(OUTPUT_ZIP)}")
    with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        count = 0
        for p in Path(OUTPUT_DIR).rglob("*"):
            if p.is_file():
                rel = p.relative_to(OUTPUT_DIR)
                zf.write(p, f"output/{rel}")
                count += 1
    output_size_mb = os.path.getsize(OUTPUT_ZIP) / 1e6
    print(f"   ✅ {count} ファイル / {output_size_mb:.1f} MB")
    print(f"   📦 {OUTPUT_ZIP}")
else:
    print(f"\nℹ️  outputなし、outputzipは作成しません")
    OUTPUT_ZIP = None

# ===== 完了表示 =====
print(f"\n🎉 バックアップ完了")
print(f"\n← Jupyter のファイルブラウザから zip を右クリック → Download してください")
print(f"  ⚠️  メインzipは次回起動時にアップロードします")
print(f"  📥 outputzipはローカル保管用です")

# ===== Slack通知 =====
if SLACK_WEBHOOK_URL:
    msg_lines = [
        f"✅ バックアップ完了",
        f"📦 メイン: {os.path.basename(MAIN_ZIP)} ({main_size_mb:.1f} MB)",
    ]
    if OUTPUT_ZIP:
        msg_lines.append(f"🖼️ output: {os.path.basename(OUTPUT_ZIP)} ({output_size_mb:.1f} MB)")
    msg_lines.append(f"📥 JupyterからDLしてからRunPodをTerminateしてください")
    
    try:
        _req.post(SLACK_WEBHOOK_URL, json={"text": "\n".join(msg_lines)}, timeout=10)
        print("\n📱 Slack通知送信済み")
    except Exception as e:
        print(f"\n⚠️  Slack通知失敗: {e}")
else:
    print("\n⚠️  SLACK_WEBHOOK_URL未設定のためSlack通知スキップ")